In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/creditcard.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, average_precision_score, precision_recall_fscore_support

df = pd.read_csv('/content/drive/MyDrive/creditcard.csv')
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape, df['Class'].sum())

X = df.drop('Class', axis=1)
y = df['Class']

# 1. split first
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. scale Time and Amount, fit on train only
cols = ['Time', 'Amount']
scaler = RobustScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[cols] = scaler.fit_transform(X_train[cols])
X_test[cols] = scaler.transform(X_test[cols])

# 3. SMOTE on the training set only
X_res, y_res = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(y_train.value_counts().to_dict(), '->', y_res.value_counts().to_dict())

# 4. train and evaluate on the untouched test set
def evaluate(model, name):
    model.fit(X_res, y_res)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, average='binary')
    print(f"{name}: precision {p:.3f} | recall {r:.3f} | F1 {f:.3f} | PR-AUC {average_precision_score(y_test, proba):.3f}")
    print(confusion_matrix(y_test, pred))

evaluate(LogisticRegression(max_iter=1000), 'Logistic Regression')
evaluate(RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42), 'Random Forest')

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 324, in run
    session = self.get_default_session(options)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/index_command.py", line 71, in get_default_session
    self._session = self.enter_context(self._build_session(options))
                                       ~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/index_command.py", line 100, in _build_session
    session = PipSession(
        cache=os.path.join(cache_dir, "http-v2") if cache_dir else None,
    ...<3 lines>...
        ssl_context=ssl_context,
    )
  Fil

In [ ]:
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

def evaluate_on(model, name, Xtr, ytr):
    model.fit(Xtr, ytr)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, average='binary')
    print(f"{name}: precision {p:.3f} | recall {r:.3f} | F1 {f:.3f} | PR-AUC {average_precision_score(y_test, proba):.3f}")
    print(confusion_matrix(y_test, pred))
    return proba

# XGBoost on the full SMOTE training set
xgb_proba = evaluate_on(
    XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                  tree_method='hist', n_jobs=-1, random_state=42),
    'XGBoost', X_res, y_res)

# SVM: a full kernel SVM is far too slow on 450K rows, so use a linear SVM
# on a 100K sample, calibrated so it gives probabilities
X_s = X_res.sample(100000, random_state=42)
y_s = y_res.loc[X_s.index]
svm_proba = evaluate_on(CalibratedClassifierCV(LinearSVC(max_iter=5000), cv=3),
                        'Linear SVM', X_s, y_s)

XGBoost: precision 0.743 | recall 0.789 | F1 0.765 | PR-AUC 0.820
[[56625    26]
 [   20    75]]
Linear SVM: precision 0.055 | recall 0.874 | F1 0.103 | PR-AUC 0.658
[[55219  1432]
 [   12    83]]


In [ ]:
import numpy as np

# validation split carved out of the TRAINING data only
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)
X_tr_res, y_tr_res = SMOTE(random_state=42).fit_resample(X_tr, y_tr)

rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_tr_res, y_tr_res)
val_proba = rf.predict_proba(X_val)[:, 1]
test_proba = rf.predict_proba(X_test)[:, 1]

def band_report(y_true, proba, low, high):
    y = np.asarray(y_true)
    auto_fraud = proba >= high
    auto_ok = proba <= low
    review = ~(auto_fraud | auto_ok)
    return {
        'low': low, 'high': high,
        'review %': round(100 * review.mean(), 2),
        'missed frauds (auto-OK)': int(((y == 1) & auto_ok).sum()),
        'false alarms (auto-fraud)': int(((y == 0) & auto_fraud).sum()),
        'frauds in review': int(((y == 1) & review).sum()),
        'genuine in review': int(((y == 0) & review).sum()),
    }

rows = [band_report(y_val, val_proba, lo, hi)
        for lo in [0.05, 0.1, 0.2, 0.3]
        for hi in [0.5, 0.7, 0.8, 0.9]]
print(pd.DataFrame(rows).to_string(index=False))

 low  high  review %  missed frauds (auto-OK)  false alarms (auto-fraud)  frauds in review  genuine in review
0.05   0.5      0.69                        8                          4                 3                309
0.05   0.7      0.69                        8                          4                 5                309
0.05   0.8      0.71                        8                          2                10                311
0.05   0.9      0.72                        8                          1                17                312
0.10   0.5      0.21                        9                          4                 2                 94
0.10   0.7      0.22                        9                          4                 4                 94
0.10   0.8      0.23                        9                          2                 9                 96
0.10   0.9      0.25                        9                          1                16                 97
0.20   0.5

In [ ]:
import joblib

low, high = 0.2, 0.8
print(pd.DataFrame([band_report(y_test, test_proba, low, high)]).to_string(index=False))

# comparison: same model, plain 0.5 threshold
print(confusion_matrix(y_test, (test_proba >= 0.5).astype(int)))

# save everything the app needs (do NOT refit the model: thresholds depend on this exact one)
joblib.dump({'model': rf, 'scaler': scaler, 'columns': list(X_train.columns),
             'low': low, 'high': high},
            '/content/drive/MyDrive/fraud_model.joblib', compress=3)

# sample transactions (raw, unscaled) for the app's "load sample" button
idx = (list(X_test[y_test == 1].sample(10, random_state=1).index)
       + list(X_test[y_test == 0].sample(20, random_state=1).index))
sample_raw = X.loc[idx].copy()
sample_raw['Class'] = y.loc[idx]
sample_raw.to_csv('/content/drive/MyDrive/sample_transactions.csv', index=False)

!ls -lh /content/drive/MyDrive/fraud_model.joblib

 low  high  review %  missed frauds (auto-OK)  false alarms (auto-fraud)  frauds in review  genuine in review
 0.2   0.8      0.07                       18                          2                14                 26
[[56642     9]
 [   23    72]]
-rw------- 1 root root 3.3M Oct  3 18:28 /content/drive/MyDrive/fraud_model.joblib


In [ ]:
import sklearn; print(sklearn.__version__)

1.6.1
